# Question 5
## a. - e.

In [ ]:
import math
import random
import textwrap
from collections import Counter, defaultdict
from typing import Sequence

from nltk.corpus import gutenberg
from nltk.tokenize import word_tokenize
from nltk.util import ngrams

UNK_TOKEN = "<UNK>"

class NGramLanguageModel:
    """A word-level n-gram language model with optional Laplace smoothing."""

    def __init__(self, n: int):
        if n < 2:
            raise ValueError("n must be at least 2 for this assignment.")

        self.n = n

        self.train_tokens: list[str] = []
        self.test_tokens: list[str] = []
        self.vocab: set[str] = set()

    def prepare_data(self, text: str) -> None:
        """Tokenize the corpus and create a sequential 90/10 train/test split.
        The vocabulary is constructed from training tokens only.
        Unseen test tokens are mapped to <UNK>.
        """
        tokens = word_tokenize(text.lower())
        split_idx = int(0.9 * len(tokens))

        self.train_tokens = tokens[:split_idx]
        self.vocab = set(self.train_tokens)
        self.vocab.add(UNK_TOKEN)

        self.test_tokens = [
            token if token in self.vocab else UNK_TOKEN
            for token in tokens[split_idx:]
        ]

    def train(self) -> None:
        """Count n-grams and the histories that have observed continuations.
        """
        if not self.train_tokens:
            raise ValueError("Call prepare_data(text) before train().")

        self.ngram_counts = Counter()
        self.history_counts = Counter()
        self.next_word_counts = defaultdict(Counter)

        for ngram in ngrams(self.train_tokens, self.n):
            history = ngram[:-1]   # The history string/tuple
            next_word = ngram[-1]  # The word following the history

            self.ngram_counts[ngram] += 1
            self.history_counts[history] += 1
            self.next_word_counts[history][next_word] += 1

    def prob(self, word: str, history: Sequence[str], smoothing=None) -> float:
        """Return P(word | history).
        Args:
            word: Candidate next token.
            history: n-1 preceding tokens.
            smoothing: None (MLE) or 'Laplace' (add-one smoothing).
        """

        history_tuple = tuple(history)
        vocab_size = len(self.vocab)

        hist_count = self.history_counts.get(history_tuple, 0)
        word_count = self.next_word_counts.get(history_tuple, {}).get(word, 0)

        if smoothing == 'Laplace':
            return (word_count + 1) / (hist_count + vocab_size)

        if hist_count == 0:
            return 0.0

        return word_count / hist_count

    def perplexity(self, tokens: Sequence[str], smoothing=None) -> float:
        """Compute perplexity using predictions with full n-1-token histories.
        """
        num_tokens = len(tokens) - (self.n - 1)
        if num_tokens <= 0:
            return float('inf')

        log_sum = 0.0 # accumulate log(prob) for numerical stability
        # skip first n - 1 tokens such that the length of history of each token in calculation is at least n - 1
        for i in range(self.n - 1, len(tokens)):
            prob = self.prob(tokens[i], tokens[i - self.n + 1: i], smoothing)
            if prob == 0:
                return float('inf')
            log_sum += math.log(prob)

        return math.exp(-log_sum / num_tokens)

    def generate(self,
        history: Sequence[str],
        max_tokens: int,
        strategy: str,
        temperature: float = 1.0,
    ) -> list[str]:
        result = []

        """Generate up to max_tokens new tokens from an initial history.
        Strategies: 'greedy', 'random', 'temperature'.
        Use MLE probabilities (no smoothing) for generation.
        Stop if the current history has no observed continuations.
        Return only the generated tokens, excluding the initial history.
        """
        if temperature <= 0.0:
            raise ValueError("Temperature must be non-negative.")

        for i in range (max_tokens):
            next_word_counts = self.next_word_counts[tuple(history)]

            if next_word_counts == {}:
                break

            if strategy == 'random':
                words, weights = zip(*next_word_counts.items())
                next_word = random.choices(words, weights=weights, k=1)[0]
            elif strategy == 'temperature':
                words, counts = zip(*next_word_counts.items())
                weights = [c ** (1.0 / temperature) for c in counts]
                next_word = random.choices(words, weights=weights, k=1)[0]
            else: # defaulting to greedy
                next_word = next_word_counts.most_common(1)[0][0]

            result.append(next_word)
            history = tuple(history[1:]) + (next_word,)

        return result

## f.

In [ ]:
    text = gutenberg.raw("carroll-alice.txt")
    random.seed(42)  # For reproducible text-generation experiments

    models: dict[int, NGramLanguageModel] = {}
    stats = []

    for n in (2, 3, 4):
        model = NGramLanguageModel(n)
        model.prepare_data(text)
        model.train()
        models[n] = model

        stats.append({
            "n": n,
            "train_tokens": len(model.train_tokens),
            "test_tokens": len(model.test_tokens),
            "vocab_size": len(model.vocab),
            "ngram_count": len(model.ngram_counts),
            "train_mle": model.perplexity(model.train_tokens, smoothing=None),
            "test_mle": model.perplexity(model.test_tokens, smoothing=None),
            "train_laplace": model.perplexity(model.train_tokens, smoothing='Laplace'),
            "test_laplace": model.perplexity(model.test_tokens, smoothing='Laplace'),
        })

    header = f"{'n':<4} | {'Train (MLE)':<12} | {'Test (MLE)':<12} | {'Train (Laplace)':<15} | {'Test (Laplace)':<15}"

    print(header)
    print("-" * len(header))
    for s in stats:
        print(
            f"{s['n']:<4} | "
            f"{s['train_mle']:<12.4f} | "
            f"{s['test_mle']:<12.4f} | "
            f"{s['train_laplace']:<15.4f} | "
            f"{s['test_laplace']:<15.4f}"
        )


### Discussion - TODO
#### Train set
For training data, higher n is associated with lower MLE perplexity. This is expected, since as the model was trained on the data, it gets surprised less as its memory increases.

#### Test set
    # TODO: doc: reason for infinite perplexity: word "speaker" appearing first on line 3219 (from 3600 total)


## g.
see implementation in class above

## h.1

In [ ]:
    generated = {}
    for (strategy, temperature) in (
            ("greedy", 1.0),
            ("random", 1.0),
            ("temperature", 0.01),
            ("temperature", 1.0),
            ("temperature", 100.0)
    ):
        result = models[3].generate(
            history=["she", "said"],
            max_tokens=100,
            strategy=strategy,
            temperature=temperature,
        )
        generated[(strategy, temperature)] = result

    # Pretty print with explicit text wrapping
    for (strategy, temperature) in generated.keys():
        temp_val = temperature if strategy == "temperature" else "N/A"
        print(f"[{strategy.upper()} | Temp: {temp_val}]")

        text = generated[(strategy, temperature)]
        # Handle both string and list-of-tokens generation outputs
        text = " ".join(str(token) for token in text)

        wrapped = textwrap.fill(text, width=80, initial_indent="  ", subsequent_indent="  ")
        print(wrapped)
        print("-" * 40)


## h.2